## 0. Setup & Config

Configure dependencies, paths, experiment settings, logging, and output folders.

| Cell | What it does | Output |
|---|---|---|
| 0.1 | Install/import/configure | EXPERIMENT, DB_BASE, logger |

$$r_t = \frac{P_t}{P_{t-1}} - 1$$

Macro analysis translates portfolio holdings into factor sensitivities. Scenario PnL helps separate tolerable volatility from true macro vulnerability.


In [ ]:

# 0.1 Silent installs, imports, EXPERIMENT, paths, logging, style
import os
import sys
import subprocess
import logging
from pathlib import Path
from datetime import datetime

required_packages = ["pandas", "numpy", "plotly", "scikit-learn", "yfinance"]
for package in required_packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    except Exception as exc:
        print(f"[WARNING] Failed to install {package}: {exc}")

import numpy as np
import pandas as pd
import plotly.express as px

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print(f"[WARNING] Drive mount skipped: {exc}")

def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    candidates.extend([
        Path("/content/ml-trading-thesis-bot"),
        Path("/content/drive/MyDrive/ml-trading-thesis-bot"),
        Path("/content/drive/MyDrive/GitHub/ml-trading-thesis-bot"),
    ])
    for candidate in candidates:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate.resolve()
    return start.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DB_BASE = Path(os.environ.get(
    "ML_TRADING_DB_BASE",
    "/Users/itsgennymac/Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/Database Finanziario" if not IN_COLAB else "/content/drive/MyDrive/Database Finanziario",
)).expanduser()
DATA_PATH = DB_BASE
os.environ["ML_TRADING_DB_BASE"] = str(DB_BASE)
os.environ["DATA_PATH"] = str(DATA_PATH)

EXPERIMENT = {
    "name": "macro_analysis",
    "target": "portfolio_exposure",
    "horizons": [5, 20, 60],
    "test_start": "2021-01-01",
    "embargo": 20,
    "n_quantiles": 5,
    "cost_bps": 10.0,
    "models": ["linear_diagnostic"],
    "run_ablation": True,
    "run_backtest": True,
    "save_figures": True,
    "feature_blocks": ["controls", "market", "risk", "signal"],
}

OUTPUT_DIR = DB_BASE / "analysis_outputs" / EXPERIMENT["name"]
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
LOGS_DIR = OUTPUT_DIR / "logs"
EXPORT_HTML_DIR = DB_BASE / "notebook_exports" / "html"
EXPORT_MARKDOWN_DIR = DB_BASE / "notebook_exports" / "markdown"
EXPORT_CSV_DIR = DB_BASE / "notebook_exports" / "csv"
EXPORT_CHARTS_DIR = DB_BASE / "notebook_exports" / "charts"
for folder in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR, LOGS_DIR, EXPORT_HTML_DIR, EXPORT_MARKDOWN_DIR, EXPORT_CSV_DIR, EXPORT_CHARTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(LOGS_DIR / "experiment.log"), logging.StreamHandler()],
    force=True,
)
logger = logging.getLogger(EXPERIMENT["name"])
np.random.seed(42)

COLORS = {
    "primary": "#01696f", "accent": "#da7101", "q1": "#c0392b", "neutral": "#7a7974",
    "bg": "#f7f6f2", "blue": "#006494", "gold": "#d19900", "purple": "#7a39bb",
}
MODEL_COLORS = {"linear_diagnostic": COLORS["primary"], "baseline": COLORS["neutral"]}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_BASE:", DB_BASE)


## 1. Data Ingestion

Run the Analysis Studio engine with real-data-first helpers and synthetic fallback when needed.

| Cell | What it does | Output |
|---|---|---|
| 1.1 | Execute engine and collect source metadata | result_df, data_source_summary |

$$X = \{x_{i,t}\}_{i=1}^{N}$$

Macro analysis translates portfolio holdings into factor sensitivities. Scenario PnL helps separate tolerable volatility from true macro vulnerability.


In [ ]:

# 1.1 Execute Analysis Studio engine
from src.analysis.macro_analysis import MacroAnalysisEngine
from src.reporting import export_analysis_report

engine = MacroAnalysisEngine()
result_df = engine.run()
result_df = result_df.copy()

source_cols = [col for col in result_df.columns if col == "source" or col.endswith("_synthetic")]
if source_cols:
    data_source_summary = result_df[source_cols].astype(str).value_counts(dropna=False).reset_index(name="N")
else:
    data_source_summary = pd.DataFrame({"series": [EXPERIMENT["name"]], "source": ["engine_output"], "N": [len(result_df)]})
source_path = TABLES_DIR / "Table_1_data_source_summary.csv"
data_source_summary.to_csv(source_path, index=False)
print("DATA SOURCE SUMMARY")
print(data_source_summary.to_string(index=False))
result_df.head()


## 2. Cleaning & Alignment

Normalize columns, timestamps, numeric values, and missingness for downstream analysis.

| Cell | What it does | Output |
|---|---|---|
| 2.1 | Clean result table | clean_df, cleaning_summary |

$$\tilde{x}_{i,t}=x_{i,t} \;\text{if observed}$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 2.1 Cleaning and alignment
clean_df = result_df.copy()
for col in clean_df.columns:
    if "date" in col.lower():
        clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")
for col in clean_df.columns:
    if clean_df[col].dtype == "object":
        clean_df[col] = clean_df[col].where(clean_df[col].notna(), "")
cleaning_summary = pd.DataFrame({
    "column": clean_df.columns,
    "missing_pct": [clean_df[col].isna().mean() for col in clean_df.columns],
    "dtype": [str(clean_df[col].dtype) for col in clean_df.columns],
    "N": len(clean_df),
})
cleaning_summary.to_csv(TABLES_DIR / "Table_cleaning_summary.csv", index=False)
print(cleaning_summary.to_string(index=False))


## 3. Feature Engineering

Build a compact feature table from numeric columns while preserving raw fields.

| Cell | What it does | Output |
|---|---|---|
| 3.1 | Create feature summary | feature_summary |

$$z(x)=\frac{x-\mu_x}{\sigma_x}$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 3.1 Feature summary
numeric_cols = clean_df.select_dtypes(include="number").columns.tolist()
feature_summary = clean_df[numeric_cols].agg(["mean", "std", "min", "max"]).T.reset_index().rename(columns={"index": "feature"}) if numeric_cols else pd.DataFrame(columns=["feature", "mean", "std", "min", "max"])
feature_summary["N"] = len(clean_df)
feature_summary.to_csv(TABLES_DIR / "Table_feature_summary.csv", index=False)
print(feature_summary.to_string(index=False))


## 4. Targets & Labels

Create target proxies and interpretable labels suitable for ranking diagnostics.

| Cell | What it does | Output |
|---|---|---|
| 4.1 | Define target and labels | labeled_df |

$$y_i = f(X_i)$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 4.1 Targets and labels
target_col = EXPERIMENT["target"] if EXPERIMENT["target"] in clean_df.columns else (numeric_cols[0] if numeric_cols else None)
labeled_df = clean_df.copy()
if target_col is not None:
    labeled_df["target_proxy"] = pd.to_numeric(labeled_df[target_col], errors="coerce")
    median_target = labeled_df["target_proxy"].median()
    labeled_df["target_label"] = np.where(labeled_df["target_proxy"] >= median_target, "high", "low")
else:
    labeled_df["target_proxy"] = 0.0
    labeled_df["target_label"] = "neutral"
label_summary = labeled_df["target_label"].value_counts().rename_axis("label").reset_index(name="N")
label_summary.to_csv(TABLES_DIR / "Table_target_labels.csv", index=False)
print("Target column:", target_col)
print(label_summary.to_string(index=False))


## 5. Descriptive Stats

Summarize sample size, numeric distributions, and coverage.

| Cell | What it does | Output |
|---|---|---|
| 5.1 | Save descriptive statistics | Table_descriptive_stats.csv |

$$\bar{x}=\frac{1}{N}\sum_i x_i$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 5.1 Descriptive statistics
desc = labeled_df.select_dtypes(include="number").describe().T.reset_index().rename(columns={"index": "variable"})
desc["N"] = len(labeled_df)
desc.to_csv(TABLES_DIR / "Table_descriptive_stats.csv", index=False)
print(desc.round(4).to_string(index=False))


## 6. Exploratory / Event Study

Render the two primary Plotly charts generated by the engine.

| Cell | What it does | Output |
|---|---|---|
| 6.1 | Display and save charts | Plotly HTML charts |

$$E[r|event] - E[r]$$

Macro analysis translates portfolio holdings into factor sensitivities. Scenario PnL helps separate tolerable volatility from true macro vulnerability.


In [ ]:

# 6.1 Plotly charts
chart_paths = []
for idx, chart in enumerate(engine.last_charts[:2], start=1):
    chart_path = EXPORT_CHARTS_DIR / f"{EXPERIMENT['name']}_chart_{idx}.html"
    chart.write_html(str(chart_path))
    chart_paths.append(chart_path)
    chart.show()
chart_table = pd.DataFrame({"chart_path": [str(path) for path in chart_paths], "N": len(labeled_df)})
chart_table.to_csv(TABLES_DIR / "Table_chart_exports.csv", index=False)
print(chart_table.to_string(index=False))


## 7. Single-Factor Diagnostics

Measure missingness and single-column relationships with the target proxy.

| Cell | What it does | Output |
|---|---|---|
| 7.1 | Run diagnostics | Table_single_factor_diagnostics.csv |

$$IC = corr(f_i, y_i)$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 7.1 Single-factor diagnostics
diag_rows = []
for col in numeric_cols:
    values = pd.to_numeric(labeled_df[col], errors="coerce")
    corr = values.corr(labeled_df["target_proxy"]) if values.notna().sum() > 2 else np.nan
    diag_rows.append({"feature": col, "missing_pct": values.isna().mean(), "target_corr": corr, "N": int(values.notna().sum())})
single_factor_diagnostics = pd.DataFrame(diag_rows).sort_values("target_corr", key=lambda s: s.abs(), ascending=False)
single_factor_diagnostics.to_csv(TABLES_DIR / "Table_single_factor_diagnostics.csv", index=False)
print(single_factor_diagnostics.round(4).to_string(index=False))


## 8. Statistical Models (regressions / econometrics)

Fit a small transparent linear diagnostic when enough numeric columns exist.

| Cell | What it does | Output |
|---|---|---|
| 8.1 | Estimate diagnostic model | Table_model_diagnostics.csv |

$$y = X\beta + \epsilon$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 8.1 Statistical model diagnostic
model_cols = [col for col in numeric_cols if col != "target_proxy" and col in labeled_df.columns]
model_rows = []
if len(model_cols) >= 1 and labeled_df["target_proxy"].notna().sum() >= 3:
    for col in model_cols[:8]:
        x = pd.to_numeric(labeled_df[col], errors="coerce")
        y = labeled_df["target_proxy"]
        valid = x.notna() & y.notna()
        beta = np.polyfit(x[valid], y[valid], 1)[0] if valid.sum() >= 3 and x[valid].nunique() > 1 else np.nan
        model_rows.append({"model": "single_factor_ols", "feature": col, "beta": beta, "N": int(valid.sum())})
model_diagnostics = pd.DataFrame(model_rows) if model_rows else pd.DataFrame({"model": ["single_factor_ols"], "feature": ["insufficient_numeric_data"], "beta": [np.nan], "N": [len(labeled_df)]})
model_diagnostics.to_csv(TABLES_DIR / "Table_model_diagnostics.csv", index=False)
print(model_diagnostics.round(4).to_string(index=False))


## 9. ML Walk-Forward

Use chronological ordering when a date column exists and summarize fold-like diagnostics.

| Cell | What it does | Output |
|---|---|---|
| 9.1 | Walk-forward style split | Table_walk_forward.csv |

$$train_t < test_t$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 9.1 Chronological walk-forward style diagnostic
date_cols = [col for col in labeled_df.columns if "date" in col.lower()]
wf_df = labeled_df.copy()
if date_cols:
    wf_df = wf_df.sort_values(date_cols[0])
fold = np.where(np.arange(len(wf_df)) < int(len(wf_df) * 0.7), "train", "test") if len(wf_df) else []
walk_forward = pd.DataFrame({
    "fold": fold,
    "target_proxy": wf_df["target_proxy"].to_numpy() if len(wf_df) else [],
})
walk_forward_summary = walk_forward.groupby("fold", as_index=False).agg(mean_target=("target_proxy", "mean"), N=("target_proxy", "size")) if not walk_forward.empty else pd.DataFrame({"fold": [], "mean_target": [], "N": []})
walk_forward_summary.to_csv(TABLES_DIR / "Table_walk_forward.csv", index=False)
print(walk_forward_summary.round(4).to_string(index=False))


## 10. Feature Ablation

Compare simple feature blocks against a baseline target proxy.

| Cell | What it does | Output |
|---|---|---|
| 10.1 | Run ablation table | Table_ablation.csv |

$$\Delta = metric_{block} - metric_{baseline}$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 10.1 Feature ablation
baseline = labeled_df["target_proxy"].mean()
ablation_rows = []
for col in numeric_cols[:10]:
    values = pd.to_numeric(labeled_df[col], errors="coerce")
    metric = values.corr(labeled_df["target_proxy"]) if values.notna().sum() > 2 else np.nan
    ablation_rows.append({"feature_block": col, "metric": metric, "delta_vs_baseline": metric - 0 if pd.notna(metric) else np.nan, "N": int(values.notna().sum())})
ablation = pd.DataFrame(ablation_rows) if ablation_rows else pd.DataFrame({"feature_block": ["baseline"], "metric": [baseline], "delta_vs_baseline": [0.0], "N": [len(labeled_df)]})
ablation.to_csv(TABLES_DIR / "Table_ablation.csv", index=False)
print(ablation.round(4).to_string(index=False))


## 11. Backtest / Strategy Evaluation

Build a lightweight strategy/evaluation proxy from the analysis ranking.

| Cell | What it does | Output |
|---|---|---|
| 11.1 | Save strategy evaluation | Table_strategy_evaluation.csv |

$$NAV_t = \prod_{s\le t}(1+r_s)$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 11.1 Strategy evaluation proxy
strategy_df = labeled_df.copy()
strategy_df["rank_signal"] = strategy_df["target_proxy"].rank(pct=True)
strategy_df["position"] = np.where(strategy_df["rank_signal"] >= 0.8, 1, np.where(strategy_df["rank_signal"] <= 0.2, -1, 0))
strategy_df["strategy_score"] = strategy_df["position"] * strategy_df["target_proxy"].fillna(0)
strategy_eval = pd.DataFrame({
    "metric": ["mean_strategy_score", "hit_rate", "active_positions"],
    "value": [strategy_df["strategy_score"].mean(), (strategy_df["strategy_score"] > 0).mean(), (strategy_df["position"] != 0).sum()],
    "N": [len(strategy_df), len(strategy_df), len(strategy_df)],
})
strategy_eval.to_csv(TABLES_DIR / "Table_strategy_evaluation.csv", index=False)
print(strategy_eval.round(4).to_string(index=False))


## 12. Interpretability

Rank the fields that most influence the analysis target proxy.

| Cell | What it does | Output |
|---|---|---|
| 12.1 | Save interpretability table | Table_interpretability.csv |

$$importance_j = |corr(x_j,y)|$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 12.1 Interpretability
interpretability = single_factor_diagnostics.copy()
if not interpretability.empty:
    interpretability["importance"] = interpretability["target_corr"].abs()
    interpretability = interpretability.sort_values("importance", ascending=False)
else:
    interpretability = pd.DataFrame({"feature": ["none"], "importance": [0.0], "N": [len(labeled_df)]})
interpretability.to_csv(TABLES_DIR / "Table_interpretability.csv", index=False)
print(interpretability.head(12).round(4).to_string(index=False))


## 13. Robustness Checks

Run subperiod, placebo, and sensitivity-style checks on available columns.

| Cell | What it does | Output |
|---|---|---|
| 13.1 | Save robustness checks | Table_VII_robustness.csv |

$$H_0: metric_{placebo} = metric_{baseline}$$

The section keeps the workflow auditable by turning intermediate assumptions into saved artifacts. This makes the analysis easier to rerun, review, and extend.


In [ ]:

# 13.1 Robustness checks
rng = np.random.default_rng(42)
robustness = pd.DataFrame({
    "check": ["full_sample", "first_half", "second_half", "placebo_shuffle", "cost_sensitivity"],
    "metric": [
        labeled_df["target_proxy"].mean(),
        labeled_df["target_proxy"].iloc[: max(1, len(labeled_df)//2)].mean(),
        labeled_df["target_proxy"].iloc[max(1, len(labeled_df)//2):].mean(),
        pd.Series(rng.permutation(labeled_df["target_proxy"].fillna(0).to_numpy())).mean(),
        labeled_df["target_proxy"].mean() - EXPERIMENT["cost_bps"] / 10000,
    ],
    "N": [len(labeled_df)] * 5,
})
robustness.to_csv(TABLES_DIR / "Table_VII_robustness.csv", index=False)
print(robustness.round(4).to_string(index=False))


## 14. Final Summary

Export the final report and print all generated artifacts.

| Cell | What it does | Output |
|---|---|---|
| 14.1 | Export report and count artifacts | final manifest |

$$Artifacts = Tables + Figures + Reports$$

Macro analysis translates portfolio holdings into factor sensitivities. Scenario PnL helps separate tolerable volatility from true macro vulnerability.


In [ ]:

# 14.1 Final export and artifact counter
manifest = export_analysis_report(engine.last_result)
summary_path = TABLES_DIR / "Table_final_manifest.csv"
pd.DataFrame([manifest]).to_csv(summary_path, index=False)

tables = sorted(TABLES_DIR.glob("*.csv"))
figures = sorted(EXPORT_CHARTS_DIR.glob(f"{EXPERIMENT['name']}_chart_*.html"))
print("EXPERIMENT COMPLETE")
print(f"   Tables : {len(tables)}")
print(f"   Figures: {len(figures)}")
print(f"   HTML   : {manifest['html_path']}")
for f in tables:
    print(f"   TABLE {f.name}")
for f in figures:
    print(f"   CHART {f.name}")
